# 🎬 From Data to Insight — MovieLens Workshop

**Welcome to the Bekk intro-to-data workshop!**

In this notebook you will work through the full data value chain — the journey every piece
of data takes before it becomes something actionable:

| # | Stage | What happens |
|---|-------|-------------|
| 1 | **Data** | Load raw files and understand what we have |
| 2 | **Structured Data** | Clean, transform and join the data into a usable shape |
| 3 | **Analytics** | Compute aggregations and spot patterns |
| 4 | **Insight** | Interpret what the patterns mean |
| 5 | **Action** | Turn the insight into something concrete |

You will use the **MovieLens small dataset**, a classic benchmark dataset from GroupLens Research
that contains ~100 000 real ratings submitted by real users.

### Dataset files
| File | Contents |
|------|----------|
| `data/movies.csv` | Movie titles and pipe-separated genres |
| `data/ratings.csv` | User ratings (0.5 – 5.0 stars) with timestamps |
| `data/tags.csv` | Free-text tags applied by users |
| `data/links.csv` | Links to IMDb and TMDb IDs |

---

### How to use this notebook
- Each section has **tasks** for you to complete.
- Cells with `# YOUR CODE HERE` are where you write your solution.
- Collapsed hint cells are there if you get stuck — try without them first!
- There is a separate **solution notebook** if you want to compare afterwards.

> ⚠️ Run cells from top to bottom. Later cells depend on variables defined earlier.


## Setup

Run this cell first — it imports all the libraries we'll need.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
from pathlib import Path

# Nicer plots
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries loaded ✅")

---
# 🗄️ Section 1 — Data: Raw Facts

> *"Data is the new oil — but like oil, it needs to be refined before it's useful."*

Before we do anything clever, we need to understand what raw data we are working with.
This is the **data engineering** starting point: load the files, look at them carefully,
and document what you see.

This stage maps to the first box in the value chain:

```
Raw files on disk  →  Loaded DataFrames  →  (we understand the shape)
```

Good questions to ask at this stage:
- How many rows and columns does each table have?
- What does each column represent?
- How do the tables relate to each other?
- Are there any obvious quality issues?


## Task 1 — Load the data

Load all four CSV files into pandas DataFrames.

> 💡 **Hint:** Use `pd.read_csv('data/filename.csv')`. The `data/` folder should be next to this notebook.


In [ ]:
# YOUR CODE HERE
movies  = ...
ratings = ...
tags    = ...
links   = ...

print("movies: ", movies.shape)
print("ratings:", ratings.shape)
print("tags:   ", tags.shape)
print("links:  ", links.shape)

## Task 2 — Explore each table

Use `.head()`, `.dtypes`, `.describe()` and `.info()` to get a feel for each DataFrame.

Take a few minutes to answer these questions in a markdown cell below:
- What does one row in `ratings` represent?
- What does the `genres` column in `movies` look like?
- What does the `timestamp` column represent?


In [ ]:
# Explore movies
movies.head()

In [ ]:
# Explore ratings
ratings.head()

In [ ]:
# Explore tags
tags.head()

In [ ]:
# Check data types and missing values for ratings
ratings.info()

**✏️ Your observations (double-click to edit):**

- One row in `ratings` represents: _..._
- The `genres` column looks like: _..._
- The `timestamp` column represents: _..._


## Task 3 — Check for missing values

Missing values can cause silent bugs later. Let's find them now.

> 💡 **Hint:** `.isnull().sum()` counts missing values per column.


In [ ]:
# Check missing values in each DataFrame
for name, df in [("movies", movies), ("ratings", ratings), ("tags", tags), ("links", links)]:
    print(f"--- {name} ---")
    # YOUR CODE HERE
    print()

## Task 4 — Understand the relationships

The four tables are linked by shared keys (`movieId`, `userId`). Draw the entity-relationship
in the markdown cell below — even just as ASCII art or a bullet list.

> 💡 **Hint:** Think about which column in `ratings` lets you look up a movie in `movies`.


**✏️ Sketch the table relationships here:**

```
movies        ratings
-------       --------
movieId  <--- movieId
title         userId
genres        rating
              timestamp
```

_Fill in where `tags` and `links` connect..._


---
# 🔧 Section 2 — Structured Data: Data Engineering

Raw data is rarely ready for analysis. In this section we act as **data engineers**:
we clean, transform and join the data so it becomes one tidy, analysis-ready table.

The key transformations we need:

1. **Extract the year** from movie titles like `"Toy Story (1995)"`
2. **Convert timestamps** from Unix epoch integers to human-readable dates
3. **Split genres** — right now they are stored as `"Action|Comedy|Drama"` in a single string
4. **Join** ratings, movies (and optionally tags) into one DataFrame

After this section you will have a clean `df` that we use for all subsequent analysis.


## Task 5 — Extract the year from movie titles

Movie titles in `movies.csv` look like `"Toy Story (1995)"`. Extract the year as a new
numeric column called `year`, and create a clean title (without the year) called `title_clean`.

> 💡 **Hint:** `Series.str.extract(r'\((\d{4})\)')` captures the four-digit year inside
> parentheses. Use `str.replace` with `regex=True` to strip it from the title.


In [ ]:
# Extract year
movies['year'] = # YOUR CODE HERE

# Create a clean title (no year in parentheses)
movies['title_clean'] = # YOUR CODE HERE

movies[['title', 'title_clean', 'year']].head(10)

## Task 6 — Convert timestamps to datetime

The `timestamp` column in `ratings` and `tags` is a Unix timestamp (seconds since 1 Jan 1970).
Convert it to a proper datetime and extract a `year` and `month` column.

> 💡 **Hint:** `pd.to_datetime(series, unit='s')` converts Unix seconds to datetime.
> Once you have a datetime column you can access `.dt.year`, `.dt.month`, etc.


In [ ]:
# Convert timestamps in ratings
ratings['date']  = # YOUR CODE HERE
ratings['year']  = # YOUR CODE HERE
ratings['month'] = # YOUR CODE HERE

ratings[['userId', 'movieId', 'rating', 'date', 'year']].head()

## Task 7 — Split genres into separate rows

Right now genres are stored as a single pipe-separated string per movie:
`"Animation|Children|Comedy"`.

Create a new DataFrame called `movies_genres` where **each row represents one
(movieId, genre) pair**. This "long" format makes it much easier to aggregate by genre later.

> 💡 **Hint 1:** `Series.str.split('|')` splits into a list.
> 💡 **Hint 2:** `DataFrame.explode('column')` turns a column of lists into individual rows.
> 💡 **Hint 3:** `"(no genres listed)"` is the placeholder for unknown genres — you may want to filter those out.


In [ ]:
# Split genres
movies_genres = (
    movies
    .assign(genre=# YOUR CODE HERE )
    .explode('genre')
    # Filter out the placeholder
    # YOUR CODE HERE
    [['movieId', 'title_clean', 'year', 'genre']]
    .reset_index(drop=True)
)

print(f"Original movies rows: {len(movies)}")
print(f"movies_genres rows:   {len(movies_genres)}")
movies_genres.head(10)

## Task 8 — Join ratings with movies

Merge `ratings` with `movies` so every rating row also contains the movie title,
genres and year. Call the result `df`.

> 💡 **Hint:** `pd.merge(left, right, on='key')` — or `left.merge(right, on='key')`.
> After merging, check the row count. Did you lose any rows? Why or why not?


In [ ]:
# Merge ratings with movies
df = # YOUR CODE HERE

print(f"Ratings rows:  {len(ratings)}")
print(f"Merged df rows: {len(df)}")

df.head()

## Task 9 — Sanity check the merged table

Before moving on, verify:
- No duplicate `(userId, movieId)` pairs (a user should only rate a movie once)
- The rating column contains only values between 0.5 and 5.0
- No unexpected nulls crept in from the join


In [ ]:
# Check for duplicate (userId, movieId) pairs
n_dupes = df.duplicated(subset=['userId', 'movieId']).sum()
print(f"Duplicate (userId, movieId) pairs: {n_dupes}")

# Check rating range
print(f"Rating min: {df['rating'].min()}, max: {df['rating'].max()}")

# Check nulls
print("\nNulls in merged df:")
print(df.isnull().sum())

---
# 📊 Section 3 — Analytics: Finding Patterns

Now the fun begins. With clean, joined data we can start asking questions and computing
aggregations. This is the work of a **data analyst** or **data scientist**.

We are looking for *patterns* — regularities in the data that might mean something.
At this stage we are not yet interpreting; we are just computing and visualising.

Some questions we will explore:
- What does the distribution of ratings look like?
- Which movies have been rated most often?
- Which genres are most popular? Most highly rated?
- How has rating activity changed over time?


## Task 10 — Distribution of ratings

Plot a histogram of all ratings. What shape does the distribution have?
Is it symmetric? Skewed? What does that tell us about how people rate?

> 💡 **Hint:** `df['rating'].value_counts().sort_index().plot(kind='bar')` gives a quick bar chart,
> or use `sns.histplot` for a smoother look.


In [ ]:
fig, ax = plt.subplots()

# YOUR CODE HERE — plot the rating distribution

ax.set_title("Distribution of Movie Ratings")
ax.set_xlabel("Rating")
ax.set_ylabel("Number of ratings")
plt.tight_layout()
plt.show()

**✏️ What do you observe?**

_Write your observations here..._


## Task 11 — Most-rated movies

Find the **top 20 most-rated movies**. These are the movies users care about most.

> 💡 **Hint:** `df.groupby('title_clean')['rating'].count()` gives you the number of
> ratings per movie. Sort and take the top 20.


In [ ]:
top_rated_count = (
    df
    # YOUR CODE HERE — group, count, sort, head
)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
# YOUR CODE HERE — horizontal bar chart
ax.set_title("Top 20 Most-Rated Movies")
ax.set_xlabel("Number of ratings")
plt.tight_layout()
plt.show()

## Task 12 — Average rating per genre

Which genres are rated most *highly* on average?
Which genres get the most ratings (popularity)?

Create a summary DataFrame `genre_stats` with columns: `genre`, `avg_rating`, `num_ratings`.

> 💡 **Hint 1:** You need to join `df` with `movies_genres` (or re-explode genres inside `df`).
> 💡 **Hint 2:** `groupby('genre').agg(avg_rating=('rating','mean'), num_ratings=('rating','count'))`.
> 💡 **Hint 3:** Consider filtering to genres with at least a few hundred ratings to avoid noise.


In [ ]:
# Create a ratings dataframe with one row per (rating, genre) pair
df_genres = df.merge(
    movies_genres[['movieId', 'genre']],
    on='movieId'
)

# Aggregate
genre_stats = (
    df_genres
    # YOUR CODE HERE
    .reset_index()
    .query("num_ratings >= 500")
    .sort_values("avg_rating", ascending=False)
)

genre_stats

In [ ]:
# Plot: avg rating by genre
fig, ax = plt.subplots(figsize=(10, 6))
# YOUR CODE HERE

ax.set_title("Average Rating by Genre")
ax.set_xlabel("Average rating")
plt.tight_layout()
plt.show()

## Task 13 — Rating volume over time

How has the number of ratings changed year by year? Are people rating more or fewer
movies over time?

> 💡 **Hint:** Group `df` by `year` and count ratings. Then plot as a line chart.
> 💡 **Hint 2:** Watch out for incomplete years at the edges of the data.


In [ ]:
yearly = (
    df
    # YOUR CODE HERE — group by year, count ratings
)

fig, ax = plt.subplots()
# YOUR CODE HERE — line plot

ax.set_title("Number of Ratings per Year")
ax.set_xlabel("Year")
ax.set_ylabel("Number of ratings")
plt.tight_layout()
plt.show()

## Task 14 — Do prolific raters rate differently?

Split users into groups by how many ratings they have submitted (e.g. < 20, 20–100, > 100).
Does the *average rating* differ between groups?

> 💡 **Hint 1:** First compute `user_counts = df.groupby('userId')['rating'].count()`.
> 💡 **Hint 2:** Use `pd.cut` to bin users into groups.
> 💡 **Hint 3:** Merge the bins back into `df` and then group by the bin.


In [ ]:
user_counts = df.groupby('userId')['rating'].count().rename('n_ratings')

# Bin users into activity groups
user_groups = pd.cut(
    user_counts,
    bins=[0, 20, 100, np.inf],
    labels=['Light (≤20)', 'Medium (21–100)', 'Heavy (>100)']
)

# Merge back and compute average rating per group
df_with_group = df.merge(user_groups.rename('user_group'), on='userId')

avg_by_group = (
    df_with_group
    # YOUR CODE HERE
)

print(avg_by_group)

---
# 💡 Section 4 — Insight: What Does It Mean?

Analytics gives us *numbers*. Insight is the step where we ask **"so what?"**

A good insight is not just a number — it is a statement that explains what a pattern
*means* and *why it matters*. It should be specific, surprising (or at least non-obvious),
and point towards an action.

**Bad insight:** "Drama has a high average rating."
**Better insight:** "Despite being the most-rated genre by far, Drama scores below Film-Noir
and Documentary — suggesting quality/quantity trade-off in the catalogue."

In this section we dig deeper and combine dimensions to find more nuanced patterns.


## Task 15 — Hidden gems vs. crowd pleasers

Not all highly-rated movies are popular, and not all popular movies are highly rated.
Create a scatter plot with **number of ratings** on the x-axis and **average rating**
on the y-axis (one point per movie). Label a few interesting outliers.

This lets us identify four quadrants:
- **Crowd pleasers** — many ratings, high average (blockbusters that deliver)
- **Hidden gems** — few ratings, high average (underrated films)
- **Polarising** — many ratings, middling average
- **Forgotten** — few ratings, low average

> 💡 **Hint 1:** Compute `movie_stats = df.groupby('title_clean')['rating'].agg(['mean','count'])`.
> 💡 **Hint 2:** Filter to movies with at least 50 ratings to reduce noise.
> 💡 **Hint 3:** `ax.annotate(text, xy=(x, y))` adds labels to specific points.


In [ ]:
movie_stats = (
    df.groupby('title_clean')['rating']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'avg_rating', 'count': 'n_ratings'})
    .query("n_ratings >= 50")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 7))

# YOUR CODE HERE — scatter plot

# Add quadrant lines at median values
ax.axvline(movie_stats['n_ratings'].median(), color='grey', linestyle='--', alpha=0.5, label='Median ratings')
ax.axhline(movie_stats['avg_rating'].median(), color='grey', linestyle=':',  alpha=0.5, label='Median avg rating')

# YOUR CODE HERE — annotate a few interesting movies

ax.set_title("Movie Popularity vs. Average Rating")
ax.set_xlabel("Number of ratings (popularity)")
ax.set_ylabel("Average rating (quality)")
ax.legend()
plt.tight_layout()
plt.show()

**✏️ What patterns do you see? Name 2–3 specific movies in each quadrant.**

_Write here..._


## Task 16 — Genre popularity over time

Which genres were dominant in different decades? Plot the share of ratings per genre
per decade as a stacked area or bar chart.

> 💡 **Hint 1:** Use `df_genres` from earlier (merged with the exploded genre table).
> 💡 **Hint 2:** `(year // 10) * 10` converts a year to its decade.
> 💡 **Hint 3:** `df.pivot_table(values, index, columns, aggfunc)` reshapes for plotting.
> 💡 **Hint 4:** Normalise each decade to 100 % to see *share* rather than absolute volume.


In [ ]:
# Add decade column
df_genres['decade'] = (df_genres['year'] // 10) * 10

# Count ratings per (decade, genre)
decade_genre = (
    df_genres
    # YOUR CODE HERE
    .reset_index()
)

# Pivot to wide format: rows=decade, columns=genre
pivot = decade_genre.pivot(index='decade', columns='genre', values='n_ratings').fillna(0)

# Normalise to % share per decade
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

# Plot — pick the top N genres for readability
top_genres = pivot_pct.sum().nlargest(8).index
pivot_pct[top_genres].plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab10')
plt.title("Genre Share of Ratings per Decade")
plt.xlabel("Decade")
plt.ylabel("Share of ratings (%)")
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## Task 17 — Summarise your three most interesting insights

Good data work ends with a clear narrative. Write 3 insights in the cell below.

Each insight should follow this structure:
**We found that [pattern]. This is interesting because [reason]. It suggests [implication].**

> 💡 **Hint:** Look back at the plots you have made. What surprised you? What would you
> want to tell a colleague over coffee?


**✏️ Your three insights:**

1. We found that ... This is interesting because ... It suggests ...

2. We found that ... This is interesting because ... It suggests ...

3. We found that ... This is interesting because ... It suggests ...


---
# 🎯 Section 5 — Action: From Insight to Value

Insights are only valuable if they lead to *action*. This is where we bridge the gap
between analysis and product/business/engineering decisions.

In the context of a movie platform, actions might include:
- Improving recommendations by understanding genre preferences
- Curating "hidden gem" collections to surface underrated films
- Deciding which genre of content to acquire more of

In this section we build a simple **content-based recommender** that uses genre overlap
to suggest similar movies — a concrete, actionable artefact from our analysis.


## Task 18 — Build a genre-based recommender

Write a function `recommend(movie_title, n=5)` that:
1. Finds a movie by (partial) title match
2. Gets its genres
3. Scores every other movie by how many genres overlap
4. Returns the top `n` movies by genre overlap, then by average rating

> 💡 **Hint 1:** Use `movies[movies['title_clean'].str.contains(title, case=False)]` to find the movie.
> 💡 **Hint 2:** Represent genres as Python `set`s and use `set intersection` for overlap counting.
> 💡 **Hint 3:** Merge with `movie_stats` (from Task 15) to rank by average rating among matches.
> 💡 **Hint 4:** Filter to movies with at least 20 ratings so obscure films don't dominate.


In [ ]:
# Precompute average ratings per movie (reuse movie_stats from Task 15 if available)
movie_stats = (
    df.groupby(['movieId', 'title_clean'])['rating']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'avg_rating', 'count': 'n_ratings'})
    .query("n_ratings >= 20")
    .reset_index()
    .merge(movies[['movieId', 'genres']], on='movieId')
)

def recommend(movie_title, n=5):
    """Return the top-n genre-similar movies with high average ratings."""
    # 1. Find the target movie
    matches = movie_stats[movie_stats['title_clean'].str.contains(movie_title, case=False)]
    if matches.empty:
        print(f"No movie found matching '{movie_title}'")
        return
    target = matches.iloc[0]
    target_genres = set(target['genres'].split('|'))
    print(f"Finding movies similar to: {target['title_clean']} ({target['genres']})")
    print("-" * 60)

    # 2. YOUR CODE HERE — compute genre overlap for every movie
    #    Add a column 'overlap' to movie_stats

    # 3. YOUR CODE HERE — filter (overlap > 0, exclude the target movie itself),
    #    sort by overlap then avg_rating, return top n

recommend("Toy Story")

## Task 19 — Stretch goal: personalised recommendations

If you have time, extend the recommender to take a **user ID** and:
1. Find the user's highest-rated movies (e.g. rated ≥ 4.0)
2. Collect all genres from those movies
3. Score every *unseen* movie by how well its genres match the user's preferences
4. Return the top recommendations

> 💡 **Hint:** A user's "genre profile" could be the weighted sum of genres by rating score.


In [ ]:
def recommend_for_user(user_id, n=10):
    """Personalised recommendations based on a user's rating history."""
    # YOUR CODE HERE
    pass

# Test with a user
recommend_for_user(user_id=1)

---
# 🏁 Wrap-Up

Well done for making it through the full data value chain!

Let's recap what you did:

| Stage | What you did |
|-------|-------------|
| **Data** | Loaded 4 CSV files, explored their structure, found quality issues |
| **Structured Data** | Extracted years, converted timestamps, exploded genres, joined tables |
| **Analytics** | Computed distributions, top-lists, genre stats, temporal trends |
| **Insight** | Identified hidden gems, genre trends, wrote narrative summaries |
| **Action** | Built a working movie recommender |

### Key pandas skills you practised
- `read_csv`, `merge`, `groupby`, `agg`, `pivot_table`
- `str.extract`, `str.split`, `str.contains`
- `pd.to_datetime`, `pd.cut`, `explode`
- `matplotlib` + `seaborn` for visualisation

### What's next?
- Try the MovieLens 1M dataset — same structure but 10× more data
- Add the `tags.csv` file to enrich the recommender with user-generated keywords
- Explore collaborative filtering (recommend based on *similar users*, not just genres)
